In [15]:
import pickle
from pathlib import Path

MODEL_PATH = Path("../models/stress_som_model_200hz_ver4_30x30.pkl")

with open(MODEL_PATH, "rb") as f:
    som_model = pickle.load(f)

print("✅ Modell laddad\n")

print("Grid:", som_model.x, "x", som_model.y)
print("Input length:", som_model.input_len)
print("Trained:", som_model.is_trained)
print("Cluster method:", som_model.cluster_method)


print("Scaler IQR:", som_model.scaler.scale_)

print("\nCluster labels:", som_model.cluster_labels)

weights = som_model.som.get_weights()
print("Weights shape:", weights.shape)

✅ Modell laddad

Grid: 30 x 30
Input length: 6
Trained: True
Cluster method: density
Scaler IQR: [0.01128422 0.00449105 0.006255   0.55811063 0.04945448 0.08787163]

Cluster labels: {0: 'Baseline', 1: 'Relax', 2: 'Stress'}
Weights shape: (30, 30, 6)


In [16]:
import pandas as pd

setup = [
    (1, "baseline", 60),
    (2, "mathtest", 168),
    (3, "mathtest_answer", 5),
    (4, "neutral", 30),
    (5, "strooptest", 180),
    (6, "strooptest_answer", 5),
    (7, "neutral", 30),
    (8, "iq_test", 300),
    (9, "iq_test_answer", 5),
    (10, "neutral", 30),
    (11, "pics", 80),
    (12, "neutral", 30),
    (13, "pics", 80),
    (14, "neutral", 30),
    (15, "pics", 80),
    (16, "neutral", 30),
    (17, "pics", 80),
    (18, "neutral", 30),
    (19, "video", 60),
    (20, "video", 60),
    (21, "video", 60),
    (22, "video", 60),
    (23, "neutral", 30),
    (24, "video", 60),
    (25, "video", 60),
    (26, "video", 60),
    (27, "video", 60),
    (28, "neutral", 30),
    (29, "video", 60),
    (30, "video", 60),
    (31, "video", 60),
    (32, "video", 60),
    (33, "neutral", 30),
    (34, "video", 60),
    (35, "video", 60),
    (36, "video", 60),
    (37, "video", 60),
]

In [17]:
def map_to_state(stimulus):
    if stimulus in ["baseline", "neutral"]:
        return "Baseline"
    elif stimulus in ["mathtest", "strooptest", "iq_test", "pics", "video"]:
        return "Stress"
    else:
        return "Relax"

In [29]:
fs = 1  # <-- ändra till din feature rate (VIKTIGT)

timeline = []

current_time = 0

for block, stim, length in setup:
    state = map_to_state(stim)
    
    samples = int(length * fs)
    
    for i in range(samples):
        timeline.append({
            "time": current_time,
            "block": block,
            "true_state": state
        })
        current_time += 1/fs

timeline_df = pd.DataFrame(timeline)

In [30]:
import pandas as pd
df_test = pd.read_csv("test_features_stroop/participant_53_stroop_windows.csv")

features = [
    "HR",
    "HRV_RMSSD",
    "HRV_SDNN",
    "SCR_Count",
    "EDA_Tonic_log",
    "EDA_Phasic_log"
]

X_test = df_test[features].values

pred_clusters = som_model.predict_cluster(X_test)


df_test["cluster"] = pred_clusters


In [28]:
df_test.head()

,HR,HRV_RMSSD,HRV_SDNN,SCR_Count,participant,time_sec,EDA_Tonic_log,EDA_Phasic_log,cluster
0,93.930531,129.720395,76.222775,1.609438,53,0,6.0,1.015204,0
1,93.847094,127.471031,75.322036,1.386294,53,1,6.0,0.450162,0
2,94.317782,128.484054,76.031452,1.386294,53,2,6.0,-1.531753,0
3,94.239582,126.797939,74.995066,1.386294,53,3,6.0,0.050399,0
4,94.610658,130.796058,77.107005,1.386294,53,4,6.0,0.379921,0


In [25]:
min_len = min(len(df_test), len(timeline_df))

df_test = df_test.iloc[:min_len].reset_index(drop=True)
timeline_df = timeline_df.iloc[:min_len].reset_index(drop=True)

print("Aligned length:", len(df_test))

Aligned length: 202


In [26]:
print("df_test:", len(df_test))
print("timeline:", len(timeline_df))

df_test: 202
timeline: 202


In [33]:
df_eval = pd.concat([df_test, timeline_df], axis=1)

df_eval.head(150)

,HR,HRV_RMSSD,HRV_SDNN,SCR_Count,participant,time_sec,EDA_Tonic_log,EDA_Phasic_log,cluster,time,block,true_state
0,93.930531,129.720395,76.222775,1.609438,53.0,0.0,6.0,1.015204,0.0,0.0,1,Baseline
1,93.847094,127.471031,75.322036,1.386294,53.0,1.0,6.0,0.450162,0.0,1.0,1,Baseline
2,94.317782,128.484054,76.031452,1.386294,53.0,2.0,6.0,-1.531753,0.0,2.0,1,Baseline
3,94.239582,126.797939,74.995066,1.386294,53.0,3.0,6.0,0.050399,0.0,3.0,1,Baseline
4,94.610658,130.796058,77.107005,1.386294,53.0,4.0,6.0,0.379921,0.0,4.0,1,Baseline
...,...,...,...,...,...,...,...,...,...,...,...,...
145,90.017454,153.367244,108.101286,1.386294,53.0,145.0,6.0,-0.022569,0.0,145.0,2,Stress
146,89.800215,159.799235,110.818010,1.386294,53.0,146.0,6.0,0.281344,2.0,146.0,2,Stress
147,90.077274,161.590928,112.750719,1.386294,53.0,147.0,6.0,0.477133,2.0,147.0,2,Stress
148,90.047549,160.177458,111.605818,1.386294,53.0,148.0,6.0,0.577835,2.0,148.0,2,Stress


In [32]:
df_eval[["stimulus", "true_state", "pred_state"]].head(150)

KeyError: "['stimulus', 'pred_state'] not in index"

In [23]:
from sklearn.metrics import confusion_matrix, classification_report

labels = ["Baseline", "Relax", "Stress"]

cm = confusion_matrix(
    df_eval["true_state"],
    df_eval["pred_state"],
    labels=labels
)

print(pd.DataFrame(cm, index=labels, columns=labels))

print("\nClassification Report:")
print(classification_report(df_eval["true_state"], df_eval["pred_state"]))

          Baseline  Relax  Stress
Baseline        60      0       0
Relax            0      0       0
Stress         100      0      42

Classification Report:
              precision    recall  f1-score   support

    Baseline       0.38      1.00      0.55        60
      Stress       1.00      0.30      0.46       142

    accuracy                           0.50       202
   macro avg       0.69      0.65      0.50       202
weighted avg       0.81      0.50      0.48       202



In [ ]:
block_eval = (
    df_eval
    .groupby(["block", "true_state"])["pred_state"]
    .value_counts(normalize=True)
    .unstack()
    .fillna(0)
)

print(block_eval)

In [ ]:
df_eval["cluster"].value_counts(normalize=True)

In [ ]:
df_eval[df_eval["cluster"] == 2][["block", "stimulus", "true_state"]]

In [ ]:
df_eval.groupby("block")["cluster"].value_counts(normalize=True)

In [ ]:
print(classification_report(df_eval["true_state"], df_eval["pred_state"]))

In [ ]:
from itertools import combinations
from sklearn.metrics import classification_report, f1_score

# true binary
df_eval["true_binary"] = df_eval["true_state"].apply(
    lambda x: "Stress" if x == "Stress" else "Not Stress"
)

clusters = [0, 1, 2]

results = []

# testa alla kombinationer
for r in range(1, len(clusters)+1):
    for combo in combinations(clusters, r):
        
        def map_pred(c):
            return "Stress" if c in combo else "Not Stress"
        
        df_eval["pred_binary"] = df_eval["cluster"].apply(map_pred)
        
        report = classification_report(
            df_eval["true_binary"],
            df_eval["pred_binary"],
            output_dict=True
        )
        
        f1 = report["Stress"]["f1-score"]
        precision = report["Stress"]["precision"]
        recall = report["Stress"]["recall"]
        
        results.append({
            "clusters_as_stress": combo,
            "f1": f1,
            "precision": precision,
            "recall": recall
        })

# sortera
results_df = pd.DataFrame(results).sort_values(by="f1", ascending=False)

results_df

In [ ]:
df_eval.groupby("cluster")[
    ["HR", "HRV_RMSSD", "SCR_Count"]
].mean()